# 머신러닝 기반 영화 리뷰 감성 분석
1. 데이터 준비 (전처리된 데이터)
2. 특징 벡터 추출 (TF-IDF)
3. 머신러닝 모델별 학습 및 평가
4. 영화 리뷰 긍부정 판단

## 1.데이터 준비 (전처리된 데이터)

In [ ]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016.csv'

# 데이터 로딩
review_df = pd.read_csv(data_filename)
review_df.head()

In [ ]:
review_df.info()

In [ ]:
# 입력 데이터와 정답 데이터 추출
review_list = list(review_df.review)
label_list = list(review_df.label)

In [ ]:
# 학습 데이터와 평가 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, trian_y, test_y = train_test_split(review_list, label_list, test_size=0.2)
len(train_X), len(test_X), len(train_y), len(test_y)

## 2. 특징 벡터 추출 : TF-IDF

In [ ]:
# 한국어 감성 분석용 tokenizer 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Verb', 'Adjctive']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

In [ ]:
# 최대 단어 수 1000개
# 학습 데이터로 Vectorizer 생성 및 학습 데이터 특징 벡터 추출
from sklearn.feature_extraction.text import TfidVectorizer

vectorizer = TfidVectorizer(tokenizer=korean_tokenizer, max_features=1000)
vectorizer.fit(train_X)
len(vectorizer.get_feature_names_out())
# vectorizer.get_feature_names_out()[:10]

In [ ]:
# 테스트 데이터 특징 벡터 추출
train_X_fv = vectorizer.transform(train_X)
print(train_X_fv.shape)
print(train_X_fv)

test_X_fv = vectorizer.transform(test_X)
print(test_X_fv.shape)

In [ ]:
# 정답 데이터 변환 (np.array)
import numpy as np
train_y = np.array(trian_y)
test_y = np.array(test_y)

In [ ]:
# 데이터 일부 확인
print(trian_y[:10])
print(test_y[:10])

## 3. 머신러닝 모델별 학습 및 평가
* 의사결정 트리
* 랜덤포레스트
* 나이브 베이즈
* 로지스틱 회귀 분석
* SVM
* Perceptron

In [ ]:
# 머신러닝 모델별 학습 성능 평가 결과 저장 준비
from sklean.tree import DecisionTreeClassfier

dtc = DecisionTreeClassfier()
dtc.fit(train_X_fv, trian_y)

## 3.1 의사결정 트리 (Decision Tree)

In [ ]:
# 학습
from sklean.tree import DecisionTreeClassfier

dtc = DecisionTreeClassfier()
dtc.fit(train_X_fv, trian_y)

In [ ]:
# 성능 평가
train_score = dtc.score(train_X_fv, train_y) * 100
test_score = dtc.score(test_X_fv, test_y)
train_score, test_score


In [ ]:
# 평가 결과 score_df에 추가
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

## 3.2 랜덤 포레스트 (Random Forrest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, trian_y)

In [ ]:
train_score = rf.score(train_X_fv, train_y) * 100
test_score = rf.score(test_X_fv, test_y) * 100
print(train_score, test_score)

In [ ]:
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

## 3.3 나이브 베이즈 (Naive Baysian)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit()

## 3.4 로지스틱 회귀 분석 (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, trian_y)

In [ ]:
train_score = lr.score(train_X_fv, train_y) * 100
test_score = lr.score(test_X_fv, test_y) * 100
print(train_score, test_score)

In [ ]:
score_df.loc['LOGISTIC'] = [train_score, test_score]
score_df

## 3.5 SVM (Support Vector Machine)

In [ ]:
from sklearn.svm import LinearSVC
svc = LinearSVC(verbose=True)
svc.fit(train_X_fv, train_y)

In [ ]:
train_score = svc.score(train_X_fv, train_y) * 100
test_score = svc.score(test_X_fv, test_y) * 100
print(train_score, test_score)

In [ ]:
score_df.loc['SVM'] = [train_score, test_score]
score_df

## 3.6 Perceptron

## 3.7 성능 비교

In [ ]:
# 평가 결과 저장 데이터 프레임 확인
score_df.sort_values(by='test', ascending=False)

## 4. 영화 리뷰 긍부정 판단
* 학습된 모델 중 선택하여 활용

In [ ]:
# 영화 리뷰감성 분석용 tokenizer 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Verb', 'Adjctive']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

# 특징 백터 추출 모델 : vectorizer

# 학습 모델 선택
sa_model = svc

In [ ]:
review = '영화가 재미있다'

# 텍스트 전처리

# 특징 벡터 추출
review_fv = vectorizer.transform([review])
# print(review_fv)

# 예측
pred = sa_model.predict(review_fv)
# print(pred)

# 예측 결과 출력
result = '긍정' if pred[0] >= 0.5 else '부정'
print(f'{review} -> {result}({pred[0]})')

In [ ]:
# 함수로 만들기
def analyze_semtiment(review):
    # 특징 벡터 추출
    review_fv = vectorizer.transform([review])
    # 학습된 모델로 예측
    pred = sa_model.predict(review_fv)
    # 예측값에 따라 결과를 생성하여 변환
    result = '긍정' if pred[0] >= 0.5 else '부정'
    return result, pred[0]
    

In [ ]:
# 함수 테스트
reviews = [
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '하품만 나온다',
    '이 영화 핵노잼 ㅠㅠ',
    '이딴게 영화냐 ㅉㅉ',
    '와 개쩐다',
    '감독 뭐하는 놈이냐',
    '정말 세계관 최강자들의 영화다'
]

for review in reviews:
    sentiment, prob = analyze_semtiment(review)
    print(f'{review} -> {result}({pred[0]})')

In [ ]:
# 문장을 입력 받아서 긍부정 판단 
review = input('>> 리뷰 입력 : ')
sentiment, prob = analyze_semtiment(review)
print(f'{review} -> {result}({pred[0]})')

In [ ]:
# 배포를 위한 모델 저장

# 특징 추출용 vectorizer
import joblib
joblib.dump(vectorizer, './model/sa_movie_vectorizer.pkl')

# 감성 분석 모델 sa_model
joblib.dump(sa_model, './model/sa_movie_predict.pkl')

In [ ]:
import joblib
class SentimentAnalyzer:
    def __init__(self, tokenizer, vectorizer_file, predict_model_file):
        self.__vectorizer = joblib.load(vectorizer_file)
        self.__vectorizer = joblib.load(predict_model_file)
        self.__tokenizer = tokenizer
    def analyze_semtiment(self, review):
        # 특징 벡터 추출
        review_fv = vectorizer.transform([review])
        # 학습된 모델로 예측
        pred = self.__predict(review_fv)
        # 예측값에 따라 결과를 생성하여 변환
        result = '긍정' if pred[0] >= 0.5 else '부정'
        return result, pred[0]
    